### Cell 1: Environment Variables & Secrets • Evaluation • Imports & Setup • LLM Client (Mistral) • Retrieval-Augmented Generation (RAG)

**What this cell does**
- Imports required libraries and sets up the runtime environment.
- Loads API keys or configuration from environment variables (avoid hardcoding secrets).
- Evaluates the system using simple metrics and/or qualitative checks.

**Key parameters / objects to notice**
- `Example` ← ====================================================

**Common pitfalls & how to avoid them**
- **Missing API keys**: Ensure your `.env` file is present and variables are set.

**How to verify it worked**
- Run 2–3 test queries; ensure returned chunks are topically relevant and not duplicates.

---


In [ ]:
"""
RAG (Retrieval-Augmented Generation) Framework Example
=====================================================

This code demonstrates the core concepts of RAG:
1. Document Storage: Storing knowledge in a searchable format
2. Retrieval: Finding relevant information based on user queries
3. Augmentation: Combining retrieved context with user questions
4. Generation: Using an LLM to produce informed responses

RAG solves the problem of LLMs having outdated or limited knowledge by
allowing them to access external, up-to-date information sources.
"""

import os
import shutil
import glob
from pathlib import Path

# Uncomment and set your actual Mistral API key
os.environ["MISTRAL_API_KEY"] = ""

### Cell 2: Code Overview

**What this cell does**
- Executes a logical step in the pipeline (see key parameters and verification below).

**Key parameters / objects to notice**
- `docs_raw` ← [

**Common pitfalls & how to avoid them**
- Keep variables well‑named and log intermediate outputs.

**How to verify it worked**
- Print key outputs (shapes, counts, sample rows) to confirm expected behavior.

---


In [2]:
# ============================================================================
# STEP 1: PREPARE SAMPLE DOCUMENTS (Knowledge Base)
# ============================================================================

print("Step 1: Preparing sample documents for our knowledge base...")

docs_raw = [
    {
        "title": "Mars Missions Overview",
        "text": (
            "Mars has been a focal point of robotic exploration. Successful missions include NASA's Perseverance rover, "
            "Curiosity, and the InSight lander. Perseverance landed in Jezero Crater in 2021 to seek signs of ancient life "
            "and collect samples for potential return to Earth. The Ingenuity helicopter demonstrated powered flight on another planet."
        )
    },
    {
        "title": "Rocket Launch Basics",
        "text": (
            "Rocket launches require careful planning: trajectory design, staging, payload fairing, and engine performance. "
            "Liquid-fueled rockets can throttle and restart, while solid-fueled boosters provide high thrust but less control. "
            "Modern rockets use staged combustion for efficiency."
        )
    },
    {
        "title": "Sample Return Concepts",
        "text": (
            "Mars Sample Return concepts involve caching samples gathered by a rover, launching them into Mars orbit, "
            "and capturing them for return to Earth. Planetary protection measures are essential to prevent contamination. "
            "This complex mission requires international cooperation and advanced robotics."
        )
    },
    {
        "title": "Space Exploration Benefits",
        "text": (
            "Space exploration drives technological innovation, creates jobs, and expands human knowledge. Technologies "
            "developed for space often find applications in medicine, communications, and materials science. "
            "The economic return on space investment is typically 7-14 dollars for every dollar spent."
        )
    }
]

print(f"Created {len(docs_raw)} sample documents")
print(f"Example document: '{docs_raw[0]['title']}'")

Step 1: Preparing sample documents for our knowledge base...
Created 4 sample documents
Example document: 'Mars Missions Overview'


### Cell 3: Evaluation • Imports & Setup • LlamaIndex • Retrieval-Augmented Generation (RAG) • Text Embeddings • Vector Store (Chroma)

**What this cell does**
- Imports required libraries and sets up the runtime environment.
- Generates vector **embeddings** for each chunk to enable semantic search.
- Indexes embeddings into a **vector store** to support fast similarity search.
- Evaluates the system using simple metrics and/or qualitative checks.

**Key parameters / objects to notice**
- `DB_DIR` ← "chroma_rag_db"
- `exist_ok` ← True)
- `documents` ← [Document(text=d["text"], metadata={"title": d["title"]}) for d in docs_raw]
- `embed_model` ← HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
- `embed_model` ← embed_model
- `chroma_client` ← chromadb.PersistentClient(path=DB_DIR)
- `chroma_collection` ← chroma_client.get_or_create_collection("rag_demo")
- `vector_store` ← ChromaVectorStore(chroma_collection=chroma_collection)
- `storage_context` ← StorageContext.from_defaults(vector_store=vector_store)
- `index` ← VectorStoreIndex.from_documents(documents, storage_context=storage_context)

**Common pitfalls & how to avoid them**
- **Embedding mismatch**: Recompute the index when you change embedding models or parameters.
- **Retriever k**: Tune `k` (e.g., 3–10) based on answer groundedness vs. latency.

**How to verify it worked**
- Inspect a few embedding vectors’ shapes and norms to ensure they’re populated.
- Run 2–3 test queries; ensure returned chunks are topically relevant and not duplicates.

---


In [3]:
# ============================================================================
# STEP 2: SET UP VECTOR DATABASE (The "Retrieval" Component)
# ============================================================================

print("\nStep 2: Setting up vector database for semantic search...")

from llama_index.core import Document, VectorStoreIndex, Settings, StorageContext
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

# Clean/prepare a local Chroma DB folder
DB_DIR = "chroma_rag_db"
if os.path.exists(DB_DIR):
    shutil.rmtree(DB_DIR)
os.makedirs(DB_DIR, exist_ok=True)

# Convert raw docs to LlamaIndex Documents
documents = [Document(text=d["text"], metadata={"title": d["title"]}) for d in docs_raw]

# Set up local embedding model (converts text to numerical vectors)
# These vectors capture semantic meaning for similarity search
embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.embed_model = embed_model

# Initialize Chroma vector store
chroma_client = chromadb.PersistentClient(path=DB_DIR)
chroma_collection = chroma_client.get_or_create_collection("rag_demo")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# Build the index (this creates embeddings and stores them)
index = VectorStoreIndex.from_documents(documents, storage_context=storage_context)
print(f"Vector index built with {len(documents)} documents")



Step 2: Setting up vector database for semantic search...


/opt/anaconda3/envs/torch311clean/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Vector index built with 4 documents


### Cell 4: Error Handling • Imports & Setup • LLM Client (Mistral) • LlamaIndex • Prompting

**What this cell does**
- Imports required libraries and sets up the runtime environment.
- Defines a **prompt** template to structure the model’s response.

**Key parameters / objects to notice**
- `llm` ← MistralAI(model="mistral-small-latest")
- `llm` ← MockLLM()

**Common pitfalls & how to avoid them**
- **Prompt drift**: Keep instructions concise; include citation requirements if teaching grounded answers.
- **Broad except**: Catch specific exceptions to avoid hiding real issues.

**How to verify it worked**
- Dry‑run with a known question and confirm the answer cites retrieved context.

---


In [4]:
# ============================================================================
# STEP 3: SET UP LLM (The "Generation" Component)
# ============================================================================

print("\nStep 3: Configuring Mistral AI for response generation...")

from llama_index.llms.mistralai import MistralAI

# Set up Mistral AI (requires MISTRAL_API_KEY environment variable)
try:
    Settings.llm = MistralAI(model="mistral-small-latest")
    print("Mistral AI configured successfully")
except Exception as e:
    print(f"Warning: Mistral AI setup failed: {e}")
    print("Make sure MISTRAL_API_KEY is set in your environment")
    # Fallback to a simple response for demonstration
    class MockLLM:
        def complete(self, prompt):
            return "Mock response: Set your MISTRAL_API_KEY to see real AI responses"
    Settings.llm = MockLLM()


Step 3: Configuring Mistral AI for response generation...
Mistral AI configured successfully


### Cell 5: Error Handling • Evaluation • Retrieval-Augmented Generation (RAG) • Text Embeddings

**What this cell does**
- Generates vector **embeddings** for each chunk to enable semantic search.
- Evaluates the system using simple metrics and/or qualitative checks.

**Key parameters / objects to notice**
- `similarity_top_k` ← 3: Retrieve top 3 most relevant documents
- `response_mode` ← "compact": Efficiently combine contexts
- `query_engine` ← index.as_query_engine(similarity_top_k=3, response_mode="compact")
- `resp` ← query_engine.query(q)
- `title` ← node.node.metadata.get("title", "N/A")

**Common pitfalls & how to avoid them**
- **Retriever k**: Tune `k` (e.g., 3–10) based on answer groundedness vs. latency.
- **Broad except**: Catch specific exceptions to avoid hiding real issues.

**How to verify it worked**
- Inspect a few embedding vectors’ shapes and norms to ensure they’re populated.
- Run 2–3 test queries; ensure returned chunks are topically relevant and not duplicates.

---


In [5]:
# ============================================================================
# STEP 4: CREATE RAG QUERY ENGINE (Combines Retrieval + Generation)
# ============================================================================

print("\nStep 4: Creating RAG query engine...")

# This is where RAG magic happens:
# 1. similarity_top_k=3: Retrieve top 3 most relevant documents
# 2. response_mode="compact": Efficiently combine contexts
query_engine = index.as_query_engine(similarity_top_k=3, response_mode="compact")

def ask(q: str):
    """
    RAG in action:
    1. Query gets converted to embeddings
    2. Vector similarity search finds relevant documents
    3. Retrieved documents provide context to the LLM
    4. LLM generates response based on both query and context
    """
    print(f"\n{'='*60}")
    print(f"QUERY: {q}")
    print('='*60)
    
    try:
        resp = query_engine.query(q)
        print(f"ANSWER: {str(resp)}")
        
        # Show which documents were retrieved (transparency)
        print(f"\nSOURCES USED:")
        for i, node in enumerate(resp.source_nodes, 1):
            title = node.node.metadata.get("title", "N/A")
            print(f"  {i}. {title}")
            
    except Exception as e:
        print(f"Error: {e}")


Step 4: Creating RAG query engine...


### Cell 6: Retrieval-Augmented Generation (RAG)

**What this cell does**
- Executes a logical step in the pipeline (see key parameters and verification below).

**Key parameters / objects to notice**
- (No obvious parameters detected; read variable names for context.)

**Common pitfalls & how to avoid them**
- Keep variables well‑named and log intermediate outputs.

**How to verify it worked**
- Print key outputs (shapes, counts, sample rows) to confirm expected behavior.

---


In [6]:
# ============================================================================
# STEP 5: DEMONSTRATE RAG IN ACTION
# ============================================================================

print("\nStep 5: Testing RAG system with various queries...")

# Test questions that require different documents
ask("Where did Perseverance land, and what is it doing there?")
ask("What are the tradeoffs between solid and liquid rockets?")
ask("How might a Mars sample return work?")
ask("What are the economic benefits of space exploration?")



Step 5: Testing RAG system with various queries...

QUERY: Where did Perseverance land, and what is it doing there?
ANSWER: Perseverance landed in Jezero Crater. It is seeking signs of ancient life and collecting samples for potential return to Earth.

SOURCES USED:
  1. Mars Missions Overview
  2. Sample Return Concepts
  3. Space Exploration Benefits

QUERY: What are the tradeoffs between solid and liquid rockets?
ANSWER: Solid rockets provide high thrust but offer less control, while liquid-fueled rockets can be throttled and restarted, offering more control but potentially less immediate thrust.

SOURCES USED:
  1. Rocket Launch Basics
  2. Space Exploration Benefits
  3. Sample Return Concepts

QUERY: How might a Mars sample return work?
ANSWER: A Mars sample return mission would involve several key steps. First, a rover would gather samples on the Martian surface. These samples would then be cached and launched into Mars orbit. Once in orbit, the samples would be captured and pr

### Cell 7: Evaluation • Retrieval-Augmented Generation (RAG) • Retriever

**What this cell does**
- Configures a **retriever** to fetch top‑k relevant chunks for a given query.
- Evaluates the system using simple metrics and/or qualitative checks.

**Key parameters / objects to notice**
- `retriever` ← index.as_retriever(similarity_top_k=2)
- `res` ← retriever.retrieve("How does a Mars sample get back to Earth?")
- `int` ← 3):
- `retriever` ← index.as_retriever(similarity_top_k=k)
- `nodes` ← retriever.retrieve(query)
- `context` ← " ".join([n.node.get_content() for n in nodes]).lower()
- `found_keywords` ← [kw for kw in expected_keywords if kw.lower() in context]
- `score` ← len(found_keywords) / max(1, len(expected_keywords))
- `test_cases` ← [
- `hits` ← keyword_recall(query, keywords)

**Common pitfalls & how to avoid them**
- **Retriever k**: Tune `k` (e.g., 3–10) based on answer groundedness vs. latency.

**How to verify it worked**
- Run 2–3 test queries; ensure returned chunks are topically relevant and not duplicates.

---


In [7]:
# ============================================================================
# STEP 6: ANALYZE RETRIEVAL QUALITY
# ============================================================================

print("\n" + "="*60)
print("STEP 6: ANALYZING RETRIEVAL QUALITY")
print("="*60)

# Direct retrieval test (without generation)
print("\nDirect retrieval test:")
retriever = index.as_retriever(similarity_top_k=2)
res = retriever.retrieve("How does a Mars sample get back to Earth?")

for i, node in enumerate(res, 1):
    print(f"\n--- Retrieved Document {i} ---")
    print(f"Title: {node.node.metadata.get('title')}")
    print(f"Content: {node.node.get_content()[:200]}...")
    print(f"Relevance Score: {node.score:.3f}")

# Keyword recall evaluation
def keyword_recall(query: str, expected_keywords: list[str], k: int = 3):
    """
    Evaluates how well our retrieval system finds relevant information
    by checking if expected keywords appear in retrieved context
    """
    retriever = index.as_retriever(similarity_top_k=k)
    nodes = retriever.retrieve(query)
    context = " ".join([n.node.get_content() for n in nodes]).lower()
    
    found_keywords = [kw for kw in expected_keywords if kw.lower() in context]
    score = len(found_keywords) / max(1, len(expected_keywords))
    
    return score, found_keywords

print(f"\nKeyword recall evaluation:")
test_cases = [
    ("Where did Perseverance land?", ["Jezero", "Perseverance", "crater"]),
    ("How do rockets work?", ["liquid", "solid", "thrust", "fuel"]),
    ("Sample return mission", ["samples", "orbit", "Earth", "return"])
]

for query, keywords in test_cases:
    score, hits = keyword_recall(query, keywords)
    print(f"Query: '{query}'")
    print(f"  Recall Score: {score:.2f} | Found: {hits}")


STEP 6: ANALYZING RETRIEVAL QUALITY

Direct retrieval test:

--- Retrieved Document 1 ---
Title: Sample Return Concepts
Content: Mars Sample Return concepts involve caching samples gathered by a rover, launching them into Mars orbit, and capturing them for return to Earth. Planetary protection measures are essential to prevent ...
Relevance Score: 0.750

--- Retrieved Document 2 ---
Title: Mars Missions Overview
Content: Mars has been a focal point of robotic exploration. Successful missions include NASA's Perseverance rover, Curiosity, and the InSight lander. Perseverance landed in Jezero Crater in 2021 to seek signs...
Relevance Score: 0.490

Keyword recall evaluation:
Query: 'Where did Perseverance land?'
  Recall Score: 1.00 | Found: ['Jezero', 'Perseverance', 'crater']
Query: 'How do rockets work?'
  Recall Score: 1.00 | Found: ['liquid', 'solid', 'thrust', 'fuel']
Query: 'Sample return mission'
  Recall Score: 1.00 | Found: ['samples', 'orbit', 'Earth', 'return']


### Cell 8: Retrieval-Augmented Generation (RAG)

**What this cell does**
- Executes a logical step in the pipeline (see key parameters and verification below).

**Key parameters / objects to notice**
- `new_documents` ← [
- `text` ← "SpaceX has revolutionized space launch with reusable rockets. The Falcon 9 can land its first stage back on Earth, d...
- `metadata` ← {"title": "Reusable Rockets", "category": "innovation"}
- `text` ← "The James Webb Space Telescope represents a new era in astronomy. Its infrared capabilities allow us to see the earl...
- `metadata` ← {"title": "James Webb Telescope", "category": "astronomy"}
- `all_documents` ← documents + new_documents
- `index` ← VectorStoreIndex.from_documents(all_documents, storage_context=storage_context)

**Common pitfalls & how to avoid them**
- Keep variables well‑named and log intermediate outputs.

**How to verify it worked**
- Print key outputs (shapes, counts, sample rows) to confirm expected behavior.

---


In [8]:
# ============================================================================
# STEP 7: EXTENDING THE KNOWLEDGE BASE
# ============================================================================

print("\n" + "="*60)
print("STEP 7: EXTENDING THE KNOWLEDGE BASE")
print("="*60)

# Add new documents to existing index
new_documents = [
    Document(
        text="SpaceX has revolutionized space launch with reusable rockets. The Falcon 9 can land its first stage back on Earth, dramatically reducing launch costs. This innovation has made space more accessible.",
        metadata={"title": "Reusable Rockets", "category": "innovation"}
    ),
    Document(
        text="The James Webb Space Telescope represents a new era in astronomy. Its infrared capabilities allow us to see the earliest galaxies and study exoplanet atmospheres in unprecedented detail.",
        metadata={"title": "James Webb Telescope", "category": "astronomy"}
    )
]

# Method 1: Use insert() for single documents
print(f"Adding {len(new_documents)} new documents to knowledge base...")
for doc in new_documents:
    index.insert(doc)

# Method 2: Alternative - Rebuild index with all documents (less efficient but works)
# all_documents = documents + new_documents
# index = VectorStoreIndex.from_documents(all_documents, storage_context=storage_context)

print("Documents added successfully!")

# Test with new knowledge
ask("What innovations has SpaceX brought to space launch?")
ask("What can the James Webb Space Telescope do?")


STEP 7: EXTENDING THE KNOWLEDGE BASE
Adding 2 new documents to knowledge base...
Documents added successfully!

QUERY: What innovations has SpaceX brought to space launch?
ANSWER: SpaceX has introduced reusable rockets to space launch, notably with the Falcon 9, which can land its first stage back on Earth. This innovation has significantly reduced launch costs, making space more accessible.

SOURCES USED:
  1. Reusable Rockets
  2. Space Exploration Benefits
  3. Rocket Launch Basics

QUERY: What can the James Webb Space Telescope do?
ANSWER: The James Webb Space Telescope can see the earliest galaxies and study exoplanet atmospheres in unprecedented detail.

SOURCES USED:
  1. James Webb Telescope
  2. Space Exploration Benefits
  3. Reusable Rockets


### Cell 9: Error Handling • Imports & Setup • Retrieval-Augmented Generation (RAG)

**What this cell does**
- Imports required libraries and sets up the runtime environment.

**Key parameters / objects to notice**
- `filepaths` ← glob.glob(str(Path(folder_path) / "*.txt"))
- `new_docs` ← []
- `encoding` ← "utf-8") as f:
- `content` ← f.read()
- `text` ← content,
- `metadata` ← {
- `folder` ← "/path/to/your/documents"
- `new_docs` ← ingest_text_files(folder)

**Common pitfalls & how to avoid them**
- **Broad except**: Catch specific exceptions to avoid hiding real issues.

**How to verify it worked**
- Print key outputs (shapes, counts, sample rows) to confirm expected behavior.

---


In [9]:
# ============================================================================
# STEP9: BATCH DOCUMENT PROCESSING EXAMPLE
# ============================================================================

def ingest_text_files(folder_path: str):
    """
    Example function to ingest all .txt files from a folder
    Useful for processing large document collections
    """
    if not os.path.exists(folder_path):
        print(f"Folder {folder_path} does not exist")
        return []
    
    filepaths = glob.glob(str(Path(folder_path) / "*.txt"))
    new_docs = []
    
    for fp in filepaths:
        try:
            with open(fp, "r", encoding="utf-8") as f:
                content = f.read()
                if content.strip():  # Only add non-empty files
                    new_docs.append(Document(
                        text=content, 
                        metadata={
                            "source": fp,
                            "filename": Path(fp).name
                        }
                    ))
        except Exception as e:
            print(f"Error reading {fp}: {e}")
    
    return new_docs

# Example usage (commented out since folder may not exist):
# folder = "/path/to/your/documents"
# new_docs = ingest_text_files(folder)
# if new_docs:
#     index.insert_documents(new_docs)
#     print(f"Successfully ingested {len(new_docs)} documents from {folder}")

print("\nRAG Framework demonstration complete!")


RAG Framework demonstration complete!
